# Phase 3 · S0 — Reproduce chainRec baseline

**Mục tiêu (S0):** train chainRec *vanilla* (chưa có edge-weight) trên Goodreads sample,
cho ra checkpoint + baseline numbers ổn định để các stage sau (S1 eval chung, S3 edge-weight,
S4 ablation) xây tiếp.

**Nền tảng:** Kaggle (GPU T4, free). Bật *Settings → Accelerator → GPU T4* và *Internet → On*.

**Đầu ra của notebook này (lưu vào `/kaggle/working` và push HF nếu có token):**
- `processed/` — data_train/val/test.npy, user_idx, item_idx, user_item_map, meta.json
- `chainrec/chainrec_<sampler>.pt` — checkpoint (uniform + stagewise)
- `chainrec/history_<sampler>.json` — loss/metric theo epoch
- `chainrec/s0_baseline.json` — bảng per-stage AUC/Recall/NDCG (mốc baseline)

> **Lưu ý phạm vi:** `SAMPLE_N_USERS = 50000` (sau filter còn ~48K). Eval trong notebook này
> là *sampled@500* (chỉ để monitor + baseline nhanh). Full-ranking eval kiểu paper sẽ được thêm ở **S1**.


## 0 · Setup — install, imports, seed, HF token

In [ ]:
!pip install implicit torch -q  # implicit chỉ cần cho các stage sau; torch thường đã có trên Kaggle

import os, json, time, pickle, random
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, asdict, field
from typing import Optional, Literal
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# HF token (tùy chọn — chỉ cần khi muốn push artifact). An toàn nếu chạy ngoài Kaggle.
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception:
    print("No HF_TOKEN (sẽ không push lên HF — vẫn lưu local /kaggle/working).")


## 1 · Config — chỉnh tất cả tham số ở một chỗ

In [ ]:
@dataclass
class DataConfig:
    raw_csv: str            = "/kaggle/working/goodreads_interactions.csv"
    out_dir: str            = "/kaggle/working/processed"
    n_stages: int           = 4
    recommend_threshold: int = 4
    min_user_inter: int     = 5
    min_item_inter: int     = 5
    require_final_stage: bool = True
    sample_n_users: Optional[int] = 50000   # ← None nếu muốn full dataset (cần High-RAM)
    sample_seed: int        = 1234
    n_val_users: int        = 5000
    n_test_users: int       = 5000

@dataclass
class ModelConfig:
    n_user: int
    n_item: int
    n_stage: int       = 4
    embed_dim: int     = 16        # 16 đủ cho sample 48K; tăng 32/64 nếu lên Colab Pro
    beta: float        = 1.0
    learn_beta: bool   = True
    l2: float          = 0.01
    lr: float          = 0.001
    batch_size: int    = 1024
    n_neg: int         = 1
    n_epochs: int      = 50
    patience: int      = 5
    sampler: Literal["uniform", "stagewise"] = "uniform"
    device: str        = DEVICE

DATA_CFG  = DataConfig()
PROC      = Path(DATA_CFG.out_dir)
CKPT_DIR  = Path("/kaggle/working/chainrec"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
HF_REPO   = "vngclinh/goodreads-preprocessed"   # nơi push artifact (nếu có token)
SAMPLERS  = ["uniform", "stagewise"]            # S0 train cả 2 như paper


## 2 · Tải `goodreads_interactions.csv` (UCSD)

File interactions gốc của UCSD (~4.1GB, ~228M dòng, cột `user_id, book_id, is_read, rating, is_reviewed`).

**Bắt buộc bật `Settings → Internet = On`** để wget tải được. Hoặc Add Data một Kaggle dataset chứa file này rồi set `DATA_CFG.raw_csv`.

> Cell dưới có kiểm tra kích thước: file < 100MB bị coi là hỏng và sẽ tải lại (tránh lỗi `EmptyDataError: No columns to parse`).


In [ ]:
RAW = "/kaggle/working/goodreads_interactions.csv"
URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_interactions.csv"
MIN_BYTES = 100 * 1024 * 1024   # file thật ~4.1GB; nếu < 100MB coi như tải hỏng/dở dang

def valid(pth):
    return os.path.exists(pth) and os.path.getsize(pth) >= MIN_BYTES

candidates = [
    DATA_CFG.raw_csv,
    "/kaggle/input/goodreads/goodreads_interactions.csv",
    "/kaggle/input/goodreads-interactions/goodreads_interactions.csv",
]
found = next((p for p in candidates if valid(p)), None)

if found is None:
    # Dọn file rỗng/dở dang từ lần chạy trước (nguyên nhân lỗi "No columns to parse")
    if os.path.exists(RAW) and not valid(RAW):
        sz = os.path.getsize(RAW)
        print(f"Xoá file hỏng: {RAW} ({sz:,} bytes)")
        os.remove(RAW)
    print("Tải goodreads_interactions.csv từ UCSD (~4GB, vài phút) — CẦN Settings -> Internet = On")
    os.system(f'wget -q --show-progress "{URL}" -O "{RAW}"')
    found = RAW if valid(RAW) else None

assert found, (
    "Chưa có goodreads_interactions.csv hợp lệ.\n"
    " - Cach 1: bat Settings -> Internet = On roi chay lai cell (tu wget tu UCSD).\n"
    " - Cach 2: Add Data mot Kaggle dataset chua file nay roi set DATA_CFG.raw_csv.\n"
    f"URL goc: {URL}"
)
DATA_CFG.raw_csv = found
print(f"Using raw_csv: {found}  ({os.path.getsize(found)/1e9:.2f} GB)")


## 3 · Data loader — dựng behavior chains (2-pass, RAM-safe)

In [ ]:
class GoodreadsChainLoader:
    def __init__(self, cfg: DataConfig):
        self.cfg = cfg
        Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
        self.user_idx, self.item_idx = {}, {}
        self.n_user = self.n_item = 0
        self.interactions = None
        self.user_item_map = defaultdict(set)

    def _first_pass_counts(self, chunksize=5_000_000):
        print("[1/4] First pass: counting...")
        uc, ic, n = defaultdict(int), defaultdict(int), 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                 usecols=["user_id","book_id","is_read","rating"],
                                 chunksize=chunksize,
                                 dtype={"user_id":np.int32,"book_id":np.int32,
                                        "is_read":np.int8,"rating":np.int8}):
            for u,c in chunk["user_id"].value_counts().items(): uc[u]+=c
            for i,c in chunk["book_id"].value_counts().items(): ic[i]+=c
            n += len(chunk); print(f"    ...{n:,} rows", end="\r")
        print(f"\n    total={n:,}, users={len(uc):,}, items={len(ic):,}")
        return uc, ic

    def _build_id_maps(self, uc, ic):
        print("[2/4] Filter + id maps...")
        vu = {u for u,c in uc.items() if c >= self.cfg.min_user_inter}
        vi = {i for i,c in ic.items() if c >= self.cfg.min_item_inter}
        print(f"    valid users={len(vu):,}, items={len(vi):,}")
        if self.cfg.sample_n_users is not None:
            rng = np.random.default_rng(self.cfg.sample_seed)
            lst = sorted(vu)
            if len(lst) > self.cfg.sample_n_users:
                vu = set(rng.choice(lst, size=self.cfg.sample_n_users, replace=False).tolist())
                print(f"    sampled {len(vu):,} users")
        self.user_idx = {u:i for i,u in enumerate(sorted(vu))}
        self.item_idx = {b:i for i,b in enumerate(sorted(vi))}
        self.n_user, self.n_item = len(self.user_idx), len(self.item_idx)
        return vu, vi

    def _second_pass_extract(self, vu, vi, chunksize=5_000_000):
        print("[3/4] Second pass: extract chains...")
        rows, n = [], 0
        for chunk in pd.read_csv(self.cfg.raw_csv,
                                 usecols=["user_id","book_id","is_read","rating"],
                                 chunksize=chunksize,
                                 dtype={"user_id":np.int32,"book_id":np.int32,
                                        "is_read":np.int8,"rating":np.int8}):
            mask = chunk["user_id"].isin(vu) & chunk["book_id"].isin(vi)
            sub = chunk[mask]
            if len(sub):
                stage = np.zeros(len(sub), dtype=np.int8)
                stage[sub["is_read"].values == 1] = 1
                rated = sub["rating"].values > 0
                stage[rated] = np.maximum(stage[rated], 2)
                recom = sub["rating"].values >= self.cfg.recommend_threshold
                stage[recom] = np.maximum(stage[recom], 3)
                u = sub["user_id"].map(self.user_idx).values
                i = sub["book_id"].map(self.item_idx).values
                rows.append(np.stack([u, i, stage], axis=1))
            n += len(chunk); print(f"    ...{n:,} rows, kept {sum(len(r) for r in rows):,}", end="\r")
        self.interactions = np.concatenate(rows).astype(np.int32)
        print(f"\n    kept {len(self.interactions):,} interactions")

    def _post_filter_and_index(self):
        print("[4/4] Post-filter (require final stage)...")
        if self.cfg.require_final_stage:
            last = self.cfg.n_stages - 1
            users_last = np.unique(self.interactions[self.interactions[:,2]==last, 0])
            self.interactions = self.interactions[np.isin(self.interactions[:,0], users_last)]
            kept = np.unique(self.interactions[:,0])
            remap = {old:new for new,old in enumerate(kept)}
            self.interactions[:,0] = np.array([remap[u] for u in self.interactions[:,0]], dtype=np.int32)
            inv = {v:k for k,v in self.user_idx.items()}
            self.user_idx = {inv[old]:new for old,new in remap.items()}
            self.n_user = len(kept)
            print(f"    {self.n_user:,} users, {len(self.interactions):,} interactions")
        for u,i,_ in self.interactions:
            self.user_item_map[int(u)].add(int(i))
        sc = np.bincount(self.interactions[:,2], minlength=self.cfg.n_stages)
        print("    stage dist: " + ", ".join(f"stage{l}={c:,}" for l,c in enumerate(sc)))

    def split_train_test(self):
        print("Splitting (holdout 1 recommend-edge per val/test user)...")
        rng = np.random.default_rng(self.cfg.sample_seed)
        last = self.cfg.n_stages - 1
        user_final = defaultdict(list)
        for idx,(u,i,s) in enumerate(self.interactions):
            if s == last: user_final[int(u)].append(idx)
        eligible = [u for u,idxs in user_final.items() if len(idxs) >= 3]
        rng.shuffle(eligible)
        n_val  = min(self.cfg.n_val_users,  len(eligible)//2)
        n_test = min(self.cfg.n_test_users, len(eligible)-n_val)
        val_idx  = [rng.choice(user_final[u]) for u in eligible[:n_val]]
        test_idx = [rng.choice(user_final[u]) for u in eligible[n_val:n_val+n_test]]
        holdout = set(val_idx) | set(test_idx)
        mask = np.ones(len(self.interactions), dtype=bool); mask[list(holdout)] = False
        self.data_train = self.interactions[mask]
        self.data_val   = self.interactions[val_idx]
        self.data_test  = self.interactions[test_idx]
        print(f"    train={len(self.data_train):,}, val={len(self.data_val):,}, test={len(self.data_test):,}")

    def build(self):
        uc, ic = self._first_pass_counts()
        vu, vi = self._build_id_maps(uc, ic)
        self._second_pass_extract(vu, vi)
        self._post_filter_and_index()
        self.split_train_test()
        return self

    def save(self):
        out = Path(self.cfg.out_dir)
        np.save(out/"data_train.npy", self.data_train)
        np.save(out/"data_val.npy",   self.data_val)
        np.save(out/"data_test.npy",  self.data_test)
        pickle.dump(self.user_idx, open(out/"user_idx.pkl","wb"))
        pickle.dump(self.item_idx, open(out/"item_idx.pkl","wb"))
        pickle.dump(dict(self.user_item_map), open(out/"user_item_map.pkl","wb"))
        json.dump({"n_user":self.n_user,"n_item":self.n_item,"n_stage":self.cfg.n_stages,
                   "config":asdict(self.cfg)}, open(out/"meta.json","w"), indent=2)
        print(f"Saved to {out}/")


In [ ]:
# Chạy loader (skip nếu đã có) — first pass trên 228M dòng mất ~vài phút
if not (PROC / "data_train.npy").exists():
    loader = GoodreadsChainLoader(DATA_CFG).build()
    loader.save()
else:
    print("Processed data đã có — skip build.")

data_train = np.load(PROC/"data_train.npy")
data_val   = np.load(PROC/"data_val.npy")
data_test  = np.load(PROC/"data_test.npy")
user_item_map = pickle.load(open(PROC/"user_item_map.pkl","rb"))
meta = json.loads((PROC/"meta.json").read_text())
print(f"n_user={meta['n_user']:,}, n_item={meta['n_item']:,}, n_stage={meta['n_stage']}")
print(f"train={len(data_train):,}, val={len(data_val):,}, test={len(data_test):,}")


## 4 · chainRec model + edgewise loss + dataset

In [ ]:
class ChainRecModel(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        lb = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta: self.log_beta = nn.Parameter(lb)
        else: self.register_buffer("log_beta", lb)
        for e in [self.user_emb, self.item_emb, self.stage_emb]: nn.init.xavier_uniform_(e.weight)
        for b in [self.b_user, self.b_item]: nn.init.zeros_(b.weight)

    @property
    def beta(self): return torch.clamp(self.log_beta.exp(), min=1.0)
    def _intention(self, u, i, l): return (self.stage_emb(l)*self.item_emb(i)*self.user_emb(u)).sum(-1)
    def _rect(self, d): b=self.beta; return F.softplus(b*d)/b

    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rect(self._intention(u, i, l_t))
        return bias + acc

    def score_all_stages(self, u, i):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        dp = torch.stack([self._rect(self._intention(
                 u, i, torch.full((B,), l, dtype=torch.long, device=u.device)))
                 for l in range(self.cfg.n_stage)], dim=1)
        suffix = dp.flip(dims=[1]).cumsum(dim=1).flip(dims=[1])
        return bias.unsqueeze(1) + suffix

    def edgewise_terms(self, u, i, l_star):
        L = self.cfg.n_stage
        s = self.score_all_stages(u, i)
        lc = l_star.clamp(0, L-1)
        s_l  = s.gather(1, lc.unsqueeze(1)).squeeze(1)
        s_n  = s.gather(1, (l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1)
        s_n  = torch.where(l_star == L-1, torch.full_like(s_n, -1e9), s_n)
        p_l  = torch.sigmoid(s_l); p_n = torch.sigmoid(s_n)
        dp   = self._rect(self._intention(u, i, lc))
        p_cap = (1.0 - torch.exp(-dp)).clamp(min=1e-8)
        return p_l, p_n, p_cap


def edgewise_loss(model, u_pos,i_pos,l_pos, u_neg,i_neg,l_neg, l2, w_pos=None):
    p_pos, _, _      = model.edgewise_terms(u_pos, i_pos, l_pos)
    _, p_next, p_cap = model.edgewise_terms(u_neg, i_neg, l_neg)
    log_pos = torch.log(p_pos.clamp(min=1e-8))
    # S0: w_pos=None → vanilla (mọi edge weight=1). S3 sẽ truyền w_pos (F3) vào đây.
    loss_pos = -(w_pos*log_pos).mean() if w_pos is not None else -log_pos.mean()
    loss_neg = -(torch.log((1-p_next).clamp(min=1e-8)) + torch.log(p_cap)).mean()
    l2_loss  = l2*(model.user_emb.weight.norm(2)**2 + model.item_emb.weight.norm(2)**2) \
               / (model.cfg.n_user + model.cfg.n_item)
    return loss_pos + loss_neg + l2_loss


class ChainDataset(Dataset):
    def __init__(self, data, user_item_map, n_item, n_neg=1, sampler="uniform", all_data=None):
        self.data=data; self.uim=user_item_map; self.n_item=n_item
        self.n_neg=n_neg; self.sampler=sampler; self.rng=np.random.default_rng(SEED)
    def _neg(self, u, l):
        pos = self.uim.get(u, set())
        for _ in range(30):
            ni = self.rng.integers(0, self.n_item)
            if ni not in pos: return int(ni), int(l)
        return int(self.rng.integers(0, self.n_item)), int(l)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        u,i,l = [int(x) for x in self.data[idx]]
        ni,nl = self._neg(u,l)
        return {"u_pos":torch.tensor(u),"i_pos":torch.tensor(i),"l_pos":torch.tensor(l),
                "u_neg":torch.tensor(u),"i_neg":torch.tensor(ni),"l_neg":torch.tensor(nl)}


## 5 · Evaluator (sampled@500 — chỉ để monitor + baseline nhanh)

> Đây là eval *tạm* để theo dõi training. **Full-ranking eval kiểu paper sẽ thêm ở S1** và
> dùng chung cho cả ALS lẫn chainRec. Recall@K ở đây = Hit@K (1 positive).


In [ ]:
class Evaluator:
    def __init__(self, model, data_test, user_item_map, n_item,
                 n_neg_eval=500, K_list=(10,20), device="cpu"):
        self.m=model; self.data=data_test; self.uim=user_item_map; self.n_item=n_item
        self.n_neg=n_neg_eval; self.K=list(K_list); self.device=device
        self.rng=np.random.default_rng(999)
    @torch.no_grad()
    def evaluate(self, target_stage=None):
        self.m.eval()
        L=self.m.cfg.n_stage
        if target_stage is None: target_stage=L-1
        hits={k:[] for k in self.K}; ndcg={k:[] for k in self.K}; aucs=[]
        for u,i_pos,l in self.data:
            u,i_pos=int(u),int(i_pos)
            if int(l)!=target_stage: continue
            pos=self.uim.get(u,set()); negs=[]; tries=0
            while len(negs)<self.n_neg and tries<self.n_neg*5:
                c=self.rng.integers(0,self.n_item)
                if c not in pos and c!=i_pos: negs.append(c)
                tries+=1
            items=np.array([i_pos]+negs, dtype=np.int64)
            u_t=torch.full((len(items),), u, dtype=torch.long, device=self.device)
            i_t=torch.tensor(items, dtype=torch.long, device=self.device)
            sc=self.m.score(u_t, i_t, target_stage).cpu().numpy()
            rank=int(np.where(np.argsort(-sc)==0)[0][0])
            aucs.append((sc[0]>sc[1:]).mean())
            for k in self.K:
                hit=int(rank<k); hits[k].append(hit)
                ndcg[k].append(1/np.log2(rank+2) if hit else 0.0)
        r={"AUC":float(np.mean(aucs))}
        for k in self.K:
            r[f"Recall@{k}"]=float(np.mean(hits[k])); r[f"NDCG@{k}"]=float(np.mean(ndcg[k]))
        return r


## 6 · Trainer

In [ ]:
class Trainer:
    def __init__(self, model, cfg, data_train, data_val, data_test, user_item_map):
        self.model=model.to(cfg.device); self.cfg=cfg; self.device=cfg.device
        self.data_val=data_val
        self.opt=torch.optim.Adam(model.parameters(), lr=cfg.lr)
        ds=ChainDataset(data_train, user_item_map, cfg.n_item,
                        n_neg=cfg.n_neg, sampler=cfg.sampler, all_data=data_train)
        self.dl=DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                           num_workers=2, pin_memory=(cfg.device=="cuda"))
        self.ev=Evaluator(model, data_test, user_item_map, cfg.n_item, device=cfg.device)

    def _val_loss(self):
        self.model.eval()
        with torch.no_grad():
            idx=np.random.permutation(len(self.data_val))[:2000]; b=self.data_val[idx]
            u=torch.tensor(b[:,0],dtype=torch.long,device=self.device)
            i=torch.tensor(b[:,1],dtype=torch.long,device=self.device)
            l=torch.tensor(b[:,2],dtype=torch.long,device=self.device)
            p,_,_=self.model.edgewise_terms(u,i,l)
            return -torch.log(p.clamp(min=1e-8)).mean().item()

    def train(self, save_path):
        best, pc, hist = float("inf"), 0, []
        print(f"{'Ep':>3} | {'train':>8} | {'val':>8} | {'AUC':>6} | {'R@10':>6} | {'N@10':>6} | {'s':>5}")
        print("-"*60)
        for ep in range(1, self.cfg.n_epochs+1):
            self.model.train(); t0=time.time(); tot=nb=0
            for b in self.dl:
                self.opt.zero_grad()
                loss=edgewise_loss(self.model,
                    b["u_pos"].to(self.device), b["i_pos"].to(self.device), b["l_pos"].to(self.device),
                    b["u_neg"].to(self.device), b["i_neg"].to(self.device), b["l_neg"].to(self.device),
                    self.cfg.l2)            # w_pos=None → vanilla
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.opt.step(); tot+=loss.item(); nb+=1
            tl=tot/nb; vl=self._val_loss(); m=self.ev.evaluate()
            print(f"{ep:>3} | {tl:>8.4f} | {vl:>8.4f} | {m['AUC']:>6.4f} | "
                  f"{m['Recall@10']:>6.4f} | {m['NDCG@10']:>6.4f} | {time.time()-t0:>4.1f}")
            hist.append({"epoch":ep,"train_loss":tl,"val_loss":vl,**m})
            if vl<best: best,pc=vl,0; torch.save(self.model.state_dict(), save_path)
            else:
                pc+=1
                if pc>=self.cfg.patience: print(f"Early stop @ epoch {ep}"); break
        self.model.load_state_dict(torch.load(save_path, map_location=self.device))
        return hist


## 7 · Train cả 2 sampler (uniform + stagewise) + lưu checkpoint/history

In [ ]:
def per_stage_table(model, data_test, user_item_map, n_item, device):
    stage_names=["shelve","read","rate","recommend"]
    out={}
    for s in range(model.cfg.n_stage):
        sd=data_test[data_test[:,2]==s]
        if len(sd)==0: continue
        m=Evaluator(model, sd, user_item_map, n_item, device=device).evaluate(target_stage=s)
        out[stage_names[s]]=m
        print(f"  {stage_names[s]:>10}: AUC={m['AUC']:.4f}  R@10={m['Recall@10']:.4f}  N@10={m['NDCG@10']:.4f}")
    return out

baseline = {}
for sampler in SAMPLERS:
    print(f"\n{'='*60}\nTRAIN chainRec — sampler={sampler}\n{'='*60}")
    cfg = ModelConfig(n_user=meta["n_user"], n_item=meta["n_item"],
                      n_stage=meta["n_stage"], sampler=sampler)
    model = ChainRecModel(cfg)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
    ckpt = str(CKPT_DIR/f"chainrec_{sampler}.pt")
    hist = Trainer(model, cfg, data_train, data_val, data_test, user_item_map).train(ckpt)
    json.dump(hist, open(CKPT_DIR/f"history_{sampler}.json","w"), indent=2)
    print(f"\nPer-stage eval (sampler={sampler}):")
    baseline[sampler] = per_stage_table(model, data_test, user_item_map, meta["n_item"], cfg.device)

json.dump(baseline, open(CKPT_DIR/"s0_baseline.json","w"), indent=2)
print("\nSaved:", list(CKPT_DIR.glob("*")))


## 8 · (Tùy chọn) Push artifact lên HuggingFace

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    for p in CKPT_DIR.glob("*"):
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"chainrec/{p.name}",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
    # processed artifacts (cần cho S1/S2)
    for name in ["data_train.npy","data_val.npy","data_test.npy","user_item_map.pkl",
                 "user_idx.pkl","item_idx.pkl","meta.json"]:
        api.upload_file(path_or_fileobj=str(PROC/name), path_in_repo=f"chainrec/processed/{name}",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
    print("Pushed to HF:", HF_REPO)
else:
    print("Bỏ qua push (không có HF_TOKEN). Artifact vẫn ở /kaggle/working.")


## 9 · DoD của S0 & bước tiếp theo

**Hoàn thành S0 khi:**
- [ ] `val_loss` giảm rồi early-stop (training hội tụ, không NaN).
- [ ] Có `s0_baseline.json` với bảng per-stage cho cả uniform & stagewise.
- [ ] Checkpoint + history + processed artifacts đã lưu (và push HF nếu có token).

**Đọc kết quả:** AUC ở stage `recommend` là mốc chính. Kỳ vọng AUC rõ ràng > 0.5 và Recall@10
hợp lý; nếu AUC ≈ 0.5 hoặc loss NaN → kiểm tra lr / dữ liệu trước khi đi tiếp.

**Tiếp theo — S1:** viết `rank_eval` (full-ranking, GPU-batched) dùng chung cho ALS & chainRec,
rồi *re-evaluate* checkpoint S0 này để có số liệu so sánh chuẩn (đánh giá chainRec ở
`target_stage=recommend` để khớp positive=rating≥4 của ALS).
